In [1]:
import csv
import pandas as pd
import re
import os
from bs4 import BeautifulSoup
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import login
from dotenv import load_dotenv
import yfinance as yf
from datetime import datetime

load_dotenv()
token = os.getenv('HF_TOKEN')
login(token)

News_data = pd.read_csv('guardian_data_since_20250601.csv',encoding='utf-8')
News_data['date'] = pd.to_datetime(News_data['webPublicationDate']).dt.strftime('%Y%m%d')
News_data.drop(columns=['webPublicationDate'], inplace=True)
News_data = News_data.iloc[:199]

#8sec


c:\Users\hmaxw\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
News_data['headline'] = None
News_data['body_text'] = None

for i, item in enumerate(News_data['fields']):
    headline = re.search(r"'headline': '(.*?)', 'body'", item)
    body = re.search(r"'body': '(.*?)'}", item)
    News_data.at[i, 'headline'] = headline.group(1)
    soup = BeautifulSoup(body.group(1))
    body_text = soup.get_text()
    News_data.at[i, 'body_text'] = body_text

#30sec

In [3]:
prosus_model= AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
prosus_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')


nlp1 = pipeline('text-classification', model=prosus_model, tokenizer=prosus_tokenizer, truncation=True, max_length=512,batch_size=32)

def sentiment(texts):
    results1 = nlp1([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
    return [ i['score'] if i['label'] == 'positive' else -i['score'] if i['label'] == 'negative' else 0 for i in results1] 

News_data['title_score_prosus'] = sentiment(News_data['headline'])
News_data['content_score_prosus'] = sentiment(News_data['body_text'])
News_data['final_score_prosus'] = News_data[['title_score_prosus','content_score_prosus']].mean(axis=1)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 17575.73it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
News_data

,fields,date,headline,body_text,title_score_prosus,content_score_prosus,final_score_prosus
0,{'headline': 'Oil and gas prices jump after Ir...,20260319,Oil and gas prices jump after Iran and Israel ...,Gas prices jumped to four-year highs and oil p...,-0.878154,-0.961177,-0.919665
1,{'headline': 'Israel launches new strikes on T...,20260323,Israel launches new strikes on Tehran as Trump...,The Israeli military said it had launched a ne...,-0.597469,-0.690185,-0.643827
2,{'headline': 'Trump’s ‘very good’ talks with I...,20260324,Trump’s ‘very good’ talks with Iran buy him ti...,There have been so many abortive rounds of dip...,0.777675,-0.611746,0.082965
3,{'headline': 'Iran war briefing: US lets India...,20260306,Iran war briefing: US lets India buy Russian o...,Iran’s deputy foreign minister warned Europea...,0.000000,-0.495950,-0.247975
4,{'headline': 'Stock market falls resume as US-...,20260305,Stock market falls resume as US-Israel war wit...,A market sell-off resumed on both sides of the...,-0.951246,-0.971622,-0.961434
...,...,...,...,...,...,...,...
194,{'headline': 'US average fuel price passes $4 ...,20260331,US average fuel price passes $4 a gallon for f...,Average US fuel prices have exceeded $4 a gall...,0.890769,-0.932935,-0.021083
195,{'headline': 'Trump’s show of force in the Mid...,20260303,Trump’s show of force in the Middle East creat...,As the US and Israel opened a new chapter of c...,-0.890597,-0.672269,-0.781433
196,{'headline': 'Starmer says ‘every lever’ will ...,20260323,Starmer says ‘every lever’ will be explored to...,Keir Starmer has promised to look at using “ev...,0.827978,0.000000,0.413989
197,{'headline': 'Middle East war creating ‘larges...,20260312,Middle East war creating ‘largest supply disru...,Oil markets are facing the “largest supply dis...,-0.924379,-0.967471,-0.945925


In [13]:
oil_volatility_index = yf.download('^OVX', start='2025-06-15', end=str(datetime.today().date()))
oil_volatility_index['avg_day_oil_volatility_index'] = oil_volatility_index[['Open','Close']].mean(axis=1)
oil_volatility_index = oil_volatility_index[['avg_day_oil_volatility_index']].reset_index()
oil_volatility_index = oil_volatility_index.rename(columns={'Date': 'date'})
oil_volatility_index.columns = ['date','avg_day_oil_volatility_index']
oil_volatility_index['date'] = oil_volatility_index['date'].astype(str).str.replace(r'-', '', regex=True)
oil_volatility_index

[*********************100%***********************]  1 of 1 completed

,date,avg_day_oil_volatility_index
0,20250616,55.134998
1,20250617,66.529999
2,20250618,70.635002
3,20250620,65.075001
4,20250623,57.825001
...,...,...
196,20260327,95.000000
197,20260330,95.250000
198,20260331,91.360001
199,20260401,90.350002


In [14]:
sentiment_by_day = pd.DataFrame(News_data.groupby('date')['final_score_prosus'].mean())
sentiment_by_day = sentiment_by_day.reset_index()
sentiment_by_day

,date,final_score_prosus
0,20250616,-0.173581
1,20250624,-0.892025
2,20260105,0.708510
3,20260205,-0.812764
4,20260219,-0.831037
5,20260224,0.033453
6,20260228,-0.491894
7,20260301,-0.455890
8,20260302,-0.599695
9,20260303,-0.561056


In [15]:
combined_sentiement_and_oilOVX = pd.merge(oil_volatility_index, sentiment_by_day, on='date', how='inner')
combined_sentiement_and_oilOVX

,date,avg_day_oil_volatility_index,final_score_prosus
0,20250616,55.134998,-0.173581
1,20250624,48.475000,-0.892025
2,20260105,29.680000,0.708510
3,20260205,54.700001,-0.812764
4,20260219,56.330000,-0.831037
5,20260224,58.485001,0.033453
6,20260302,71.420002,-0.599695
7,20260303,75.770000,-0.561056
8,20260304,77.064999,-0.495787
9,20260305,81.360001,-0.445948
